In [1]:
import pandas as pd
import numpy as np

from sentence_transformers import SentenceTransformer
from sklearn.decomposition import PCA

import plotly.express as px

/home/sachin/Desktop/rs/Recommendation-system/rsvenv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ratings = pd.read_csv(
    "ml-1m/ratings.dat",
    sep="::",
    engine="python",
    names=["userId","movieId","rating","timestamp"]
)

movies = pd.read_csv(
    "ml-1m/movies.dat",
    sep="::",
    engine="python",
    encoding="latin-1",
    names=["movieId", "title", "genres"],
    header=None
)

print(ratings.shape)
print(movies.shape)
movies.head()

(1000209, 4)
(3883, 3)


,movieId,title,genres
0,1,Toy Story (1995),Animation|Children's|Comedy
1,2,Jumanji (1995),Adventure|Children's|Fantasy
2,3,Grumpier Old Men (1995),Comedy|Romance
3,4,Waiting to Exhale (1995),Comedy|Drama
4,5,Father of the Bride Part II (1995),Comedy


In [3]:
movies["text"] = movies["title"] + " " + movies["genres"].str.replace("|"," ")

In [4]:
model = SentenceTransformer("all-MiniLM-L6-v2")

movie_embeddings = model.encode(
    movies["text"],
    show_progress_bar=True
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 4924.52it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 122/122 [00:01<00:00, 67.53it/s] 


In [5]:
pca = PCA(n_components=3)

movie_coords = pca.fit_transform(movie_embeddings)

In [6]:
viz_df = pd.DataFrame(movie_coords, columns=["x","y","z"])

viz_df["movieId"] = movies["movieId"]
viz_df["title"] = movies["title"]
viz_df["genres"] = movies["genres"]

viz_df.head()

,x,y,z,movieId,title,genres
0,0.266428,0.193829,0.020326,1,Toy Story (1995),Animation|Children's|Comedy
1,-0.061311,0.089954,0.066497,2,Jumanji (1995),Adventure|Children's|Fantasy
2,0.358665,0.084968,0.191123,3,Grumpier Old Men (1995),Comedy|Romance
3,0.287901,-0.143637,0.151489,4,Waiting to Exhale (1995),Comedy|Drama
4,0.398480,0.019997,0.140946,5,Father of the Bride Part II (1995),Comedy


In [7]:
selected_users = [1, 25, 100, 300]

In [8]:
viz_df["user"] = "other"

for user in selected_users:
    
    liked_movies = ratings[
        (ratings["userId"] == user) &
        (ratings["rating"] >= 4)
    ]["movieId"]

    viz_df.loc[viz_df["movieId"].isin(liked_movies), "user"] = f"user_{user}"

In [9]:
fig = px.scatter_3d(
    viz_df,
    x="x",
    y="y",
    z="z",
    color="user",
    hover_name="title",
    opacity=0.8
)

fig.show()

In [11]:
from sklearn.cluster import KMeans

In [21]:
kmeans= KMeans(n_clusters=5, random_state=42)

clusters_3 = kmeans.fit_predict(movie_embeddings)

viz_df["cluster_3"] = clusters_3

In [22]:
fig = px.scatter_3d(
    viz_df,
    x="x",
    y="y",
    z="z",
    color="cluster_3",
    hover_name="title",
    opacity=0.8
)

fig.show()

from what i noticed users have different favourite movies in different genres and i think it's hard to recommend using content-based. it more easy to use collabrative filtering where we try to match the user with similar users.

Content-based filtering struggles when users have diverse interests across multiple genres. Collaborative filtering overcomes this limitation by learning latent user–item interactions from collective behavior, enabling recommendations that are not restricted to explicit content similarity.